# Q1c: Experiment with at least 2 other outlier smoothing or missing value imputation strategies

**INSTRUCTIONS TO USER:**
1. Use the same `parkingLot.csv`.
2. Apply KNNImputer and RobustScaler (or Moving Median) to the preprocessing pipelines of Q1a and Q1b.

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from scipy.ndimage import median_filter
from sklearn.metrics import mean_absolute_percentage_error

In [2]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from scipy.ndimage import median_filter
from sklearn.metrics import mean_absolute_percentage_error
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings("ignore")

# --- 1. Base Data Loading (From Q1a) ---
df = pd.read_csv('parkingLot.csv', dtype={'camera_id': str})
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df[df['camera_id'] == '001']
df = df[df['timestamp'].dt.hour >= 5]
df = df.dropna(subset=['timestamp'])

# Aggregate by day
daily_counts = df.set_index('timestamp').resample('D').size().astype(float)

# (Optional) Injecting a couple of artificial NaNs so KNNImputer actually has something to do!
daily_counts.iloc[2] = np.nan
daily_counts.iloc[5] = np.nan

# --- 2. Strategy 2: KNN Imputation ---
imputer = KNNImputer(n_neighbors=3)
# KNN expects a 2D array, so we reshape, impute, and flatten back to 1D
imputed_array = imputer.fit_transform(daily_counts.values.reshape(-1, 1))
imputed_counts = pd.Series(imputed_array.flatten(), index=daily_counts.index)

# --- 3. Strategy 1: Moving Median Smoothing ---
# Applies a rolling median of window size 3 to flatten out anomalous weekend spikes
smoothed_array = median_filter(imputed_counts.values, size=3)
smoothed_counts = pd.Series(smoothed_array, index=imputed_counts.index)

# --- 4. Train/Test Split ---
train = smoothed_counts[:-7]
test = smoothed_counts[-7:]

# --- 5. Modeling (ARIMA) ---
model = ARIMA(train, order=(7,1,1))
model_fit = model.fit()
forecast = model_fit.forecast(steps=7)

# --- 6. Evaluation ---
mape = mean_absolute_percentage_error(test, forecast)
print(f"MAPE (Smoothed & Imputed): {mape:.4f}")

def mase(y_true, y_pred, y_train):
    n = len(y_train)
    d = np.abs(np.diff(y_train)).sum() / (n - 1)
    errors = np.abs(y_true - y_pred)
    return errors.mean() / d

print(f"MASE (Smoothed & Imputed): {mase(test, forecast, train):.4f}")


MAPE (Smoothed & Imputed): 0.0675
MASE (Smoothed & Imputed): 3.5638
